In [ ]:
# --- 7.2 Fine-Tune Random Forest ---
print("\nFine-tuning Random Forest...")
param_grid_rf = {
    'n_estimators': [300, 500],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2]
}
grid_rf = GridSearchCV(RandomForestClassifier(random_state=42, n_jobs=-1),
                       param_grid=param_grid_rf, cv=cv, scoring='accuracy', n_jobs=-1)
try:
  grid_rf.fit(X_train, y_train)
  best_rf = grid_rf.best_estimator_
except KeyboardInterrupt:
  print("\nExecution interrupted by user.")
  # If interrupted, you might want to handle best_rf differently or raise the exception.
  # For now, let's assume it completes successfully or best_rf remains unfitted as per original error.
  # If you want to use a default RF in case of interruption, it should be fitted here.

# The line below caused the error by overwriting the fitted 'best_rf'
# best_rf = RandomForestClassifier(random_state=42)

tuned_models['Random Forest'] = best_rf
tuned_results_list.append(evaluate_model(best_rf, X_test, y_test, "Tuned Random Forest"))

In [ ]:
# --- 7.3 Fine-Tune LightGBM ---
print("\nFine-tuning LightGBM...")
param_grid_lgbm = {
    'n_estimators': [300, 500],
    'max_depth': [6, 8, 10],
    'learning_rate': [0.05, 0.1],
    'subsample': [0.8, 1.0]
}
grid_lgbm = GridSearchCV(LGBMClassifier(objective='binary', random_state=42, verbose=-1),
                         param_grid=param_grid_lgbm, cv=cv, scoring='accuracy', n_jobs=-1)
grid_lgbm.fit(X_train, y_train)
best_lgbm = grid_lgbm.best_estimator_
tuned_models['LightGBM'] = best_lgbm
tuned_results_list.append(evaluate_model(best_lgbm, X_test, y_test, "Tuned LightGBM"))

In [ ]:

# --- 7.4 Fine-Tune XGBoost ---
print("\nFine-tuning XGBoost...")
param_grid_xgb = {
    'n_estimators': [300, 500],
    'max_depth': [6, 8, 10],
    'learning_rate': [0.05, 0.1],
    'subsample': [0.8, 0.9],
    'colsample_bytree': [0.8, 0.9]
}
grid_xgb = GridSearchCV(XGBClassifier(objective='binary:logistic', random_state=42, eval_metric='logloss'),
                        param_grid=param_grid_xgb, cv=cv, scoring='accuracy', n_jobs=-1)
grid_xgb.fit(X_train, y_train)
best_xgb = grid_xgb.best_estimator_
tuned_models['XGBoost'] = best_xgb
tuned_results_list.append(evaluate_model(best_xgb, X_test, y_test, "Tuned XGBoost"))

In [ ]:
# 8. Compare Tuned Models

print("\n" + "="*60)
print("Final Comparison of Tuned Models")
print("="*60)

tuned_results_df = pd.DataFrame(
    tuned_results_list,
    index=['Tuned LogisticReg', 'Tuned RandomForest', 'Tuned LightGBM', 'Tuned XGBoost']
)
print(tuned_results_df.round(4))

# Identify the absolute best model overall
best_model_name = tuned_results_df['accuracy'].idxmax()
print(f"\n--> The best performing model after fine-tuning is: {best_model_name} with Accuracy: {tuned_results_df.loc[best_model_name, 'accuracy']:.4f}")

# Threshold tuning plot for the BEST model overall (Assuming it's XGBoost or LightGBM)
best_overall_model = tuned_models[best_model_name.replace('Tuned ', '')]
y_proba_best = best_overall_model.predict_proba(X_test)[:, 1]
thresholds = np.arange(0.1, 0.9, 0.05)
recall0 = []
precision1 = []

for thresh in thresholds:
    y_pred_thresh = (y_proba_best >= thresh).astype(int)
    recall0.append(recall_score(y_test, y_pred_thresh, pos_label=0))
    precision1.append(precision_score(y_test, y_pred_thresh, pos_label=1))

In [ ]:
plt.figure(figsize=(8,5))
plt.plot(thresholds, recall0, marker='o', label='Recall (class 0 – unsafe)')
plt.plot(thresholds, precision1, marker='s', label='Precision (class 1 – safe)')
plt.xlabel('Threshold (probability for class 1)')
plt.ylabel('Score')
plt.title(f'Threshold Tuning for Safety ({best_model_name})')
plt.legend()
plt.grid(True)
plt.show()